# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click 'Run Cell' button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

In [7]:
import os
os.environ["KAGGLEHUB_CACHE"] = "E:/kagglehub_cache"  # 你也可以换成其它路径

import kagglehub

# Download latest version
path = kagglehub.dataset_download("ahmadrazakashif/housing-detail")

print("Path to dataset files:", path)


Path to dataset files: E:/kagglehub_cache\datasets\ahmadrazakashif\housing-detail\versions\1


In [8]:
import pandas as pd
import os

file_path = os.path.join(path, "housing.csv")
house_reviews = pd.read_csv(file_path)      # Read path
print(house_reviews.head())

      price  area  bedrooms  bathrooms  stories mainroad guestroom basement  \
0  13300000  7420         4          2        3      yes        no       no   
1  12250000  8960         4          4        4      yes        no       no   
2  12250000  9960         3          2        2      yes        no      yes   
3  12215000  7500         4          2        2      yes        no      yes   
4  11410000  7420         4          1        2      yes       yes      yes   

  hotwaterheating airconditioning  parking prefarea furnishingstatus  
0              no             yes        2      yes        furnished  
1              no             yes        3       no        furnished  
2              no              no        2      yes   semi-furnished  
3              no             yes        3      yes        furnished  
4              no             yes        2       no        furnished  


In [9]:
from sklearn.model_selection import train_test_split

y = house_reviews.price
X = house_reviews.drop(['price'], axis=1)

X_train_full, X_valid_full, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

cat_cols = [col_name for col_name in X_train_full.columns if X_train_full[col_name].nunique() < 10 and X_train_full[col_name].dtype == object]
print(cat_cols)

num_cols = [col_name for col_name in X_train_full.columns if X_train_full[col_name].dtype in ['int64', 'float64']]
print(num_cols)

my_cols = num_cols + cat_cols
X_train = X_train_full[my_cols].copy()
X_valid = X_valid_full[my_cols].copy()

['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea', 'furnishingstatus']
['area', 'bedrooms', 'bathrooms', 'stories', 'parking']


In [10]:
#Importing
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

#Setting num & cat transformer
num_transformer = SimpleImputer(strategy='mean')
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

#Bundle of preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ]
)

In [11]:
#setting model
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=500, random_state=42)

my_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ]
)
my_pipeline.fit(X_train, y_train)
y_pred = my_pipeline.predict(X_valid)

In [12]:
#Validation
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(y_valid, y_pred)
print(mae)

1014868.0365443425
